In [10]:
from langchain.chat_models import init_chat_model
from langchain_postgres import PGVector
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain.tools import tool
from dotenv import load_dotenv
import os

load_dotenv()

True

In [11]:
if not os.environ.get("GOOGLE_API_KEY"):
  print("Please set the GOOGLE_API_KEY environment variable.")

model = init_chat_model("google_genai:gemini-2.5-flash")

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

In [12]:
if not os.environ.get("SUPABASE_CONNECTION_STRING"):
    print("Please set the SUPABASE_CONNECTION_STRING environment variable.")

vector_store = PGVector(
    embeddings=embeddings,
    collection_name="lilian_weng",
    connection=os.environ["SUPABASE_CONNECTION_STRING"],
)

In [13]:
@tool(response_format="content_and_artifact")
def retrieve_context(query: str):
    """Retrieve information to help answer a query."""
    retrieved_docs = vector_store.similarity_search(query, k=2)
    serialized = "\n\n".join(
        (f"Source: {doc.metadata}\nContent: {doc.page_content}")
        for doc in retrieved_docs
    )
    return serialized, retrieved_docs

In [14]:
from langchain.agents import create_agent


tools = [retrieve_context]
# If desired, specify custom instructions
prompt = (
    "You have access to a tool that retrieves context from a blog post. "
    "Use the tool to help answer user queries."
)
agent = create_agent(model, tools, system_prompt=prompt)

In [17]:
query = (
    "What are some challanges and limitations when building LLM based agents?"
)

for event in agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    stream_mode="values",
):
    last_message = event["messages"][-1]
    last_message.pretty_print()

================================ Human Message =================================

What are some challanges and limitations when building LLM based agents?
================================== Ai Message ==================================
Tool Calls:
  retrieve_context (b51dc8b2-1238-4f75-99f4-17dcbcb69eb6)
 Call ID: b51dc8b2-1238-4f75-99f4-17dcbcb69eb6
  Args:
    query: challenges and limitations of LLM based agents
================================= Tool Message =================================
Name: retrieve_context

Source: {'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'start_index': 39137}
Content: Finite context length: The restricted context capacity limits the inclusion of historical information, detailed instructions, API call context, and responses. The design of the system has to work with this limited communication bandwidth, while mechanisms like self-reflection to learn from past mistakes would benefit a lot from long or infinite context windows. Althou

In [19]:
# Render the agent's last message as Markdown in the Jupyter output
from IPython.display import Markdown, display
display(Markdown(last_message.text))

Building LLM-based agents presents several challenges and limitations:

*   **Finite context length:** LLMs have a restricted context capacity, which limits the amount of historical information, detailed instructions, API call context, and responses they can process at once. This constraint hinders mechanisms like self-reflection, which benefit greatly from longer context windows. While vector stores and retrieval can expand the accessible knowledge, their representational power is not as robust as full attention.
*   **Challenges in long-term planning and task decomposition:** LLMs struggle with planning over extended histories and effectively exploring the solution space. They also demonstrate difficulty in adjusting plans when unexpected errors arise, making them less robust compared to humans who can learn through trial and error.